# SHAP Interpretability Analysis

This notebook explains the saved XGBoost cardiovascular risk model with SHAP. Although the SVM had the highest held-out AUC in this run, XGBoost is the best classical model for transparent tree-based SHAP explanations and is therefore used for the project interpretability contribution.

In [ ]:
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.metrics import confusion_matrix

PROJECT_ROOT = Path('..').resolve() if Path('../src').exists() else Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from preprocess import split_preprocess_smote

MODELS_DIR = PROJECT_ROOT / 'models' / 'saved'
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

shap.initjs()

## Load Model And Test Data

We reload the exact saved XGBoost artifact from Phase 2 and recreate the same held-out test split. The SHAP values are computed on the transformed test matrix, so every explanation corresponds to the features seen by the model.

In [ ]:
artifact = joblib.load(MODELS_DIR / 'xgboost.pkl')
model = artifact['model']
feature_names = artifact['feature_names']
processed = split_preprocess_smote(dataset=artifact.get('dataset', 'heart'), use_smote=True)

X_test = pd.DataFrame(processed.X_test, columns=feature_names)
y_test = processed.y_test.reset_index(drop=True)
y_score = model.predict_proba(X_test)[:, 1]
y_pred = (y_score >= 0.5).astype(int)

print('X_test shape:', X_test.shape)
print('Confusion matrix [[TN, FP], [FN, TP]]:')
print(confusion_matrix(y_test, y_pred))
display(X_test.head())

## Compute SHAP Values

SHAP estimates how much each feature pushes an individual prediction higher or lower than the model's average prediction. Positive SHAP values increase predicted risk; negative values decrease predicted risk.

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values_raw = explainer.shap_values(X_test)

if isinstance(shap_values_raw, list):
    shap_values = shap_values_raw[1]
else:
    shap_values = shap_values_raw

base_value = explainer.expected_value
if isinstance(base_value, (list, np.ndarray)):
    base_value = base_value[1]

shap_explanation = shap.Explanation(
    values=shap_values,
    base_values=np.repeat(base_value, X_test.shape[0]),
    data=X_test.values,
    feature_names=feature_names,
)

print('SHAP matrix shape:', shap_values.shape)
print('Base value:', base_value)

## SHAP Summary Plot

The beeswarm plot shows the distribution of feature effects across patients. Features near the top have the largest overall influence. Points to the right raise model-estimated disease risk, while points to the left lower it.

In [ ]:
shap.summary_plot(shap_values, X_test, feature_names=feature_names, show=False)
plt.title('SHAP Summary: XGBoost CVD Risk Model')
plt.tight_layout()
plt.show()

## Mean Absolute SHAP Importance

This bar chart ranks features by average absolute SHAP value. Clinically, this is a global risk-factor ranking: it tells us which measurements the model relies on most often, not whether a feature is always harmful or protective.

In [ ]:
shap.summary_plot(shap_values, X_test, feature_names=feature_names, plot_type='bar', show=False)
plt.title('Global Feature Importance by Mean |SHAP|')
plt.tight_layout()
plt.show()

importance = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False)

display(importance.head(15))
importance.to_csv(RESULTS_DIR / 'shap_global_importance.csv', index=False)

## Individual Waterfall Plots

Waterfall plots explain a single prediction. They start from the model's average output and then add or subtract feature contributions until reaching the patient-specific prediction.

In [ ]:
def first_index(mask):
    matches = np.flatnonzero(mask)
    return int(matches[0]) if len(matches) else None

cases = {
    'True positive': first_index((y_test.values == 1) & (y_pred == 1)),
    'True negative': first_index((y_test.values == 0) & (y_pred == 0)),
    'False positive': first_index((y_test.values == 0) & (y_pred == 1)),
}

for label, idx in cases.items():
    if idx is None:
        print(f'{label}: no example in this test split')
        continue
    print(f'{label} example index={idx}, true={y_test.iloc[idx]}, predicted={y_pred[idx]}, probability={y_score[idx]:.3f}')
    shap.plots.waterfall(shap_explanation[idx], max_display=12, show=False)
    plt.title(label)
    plt.tight_layout()
    plt.show()

## Dependence Plots For Top Features

Dependence plots show how a feature's value relates to its SHAP contribution. They help answer clinical questions like whether risk rises steadily with a measurement or changes only after a threshold.

In [ ]:
top_features = importance['feature'].head(3).tolist()
print('Top features:', top_features)

for feature in top_features:
    shap.dependence_plot(feature, shap_values, X_test, feature_names=feature_names, show=False)
    plt.title(f'SHAP Dependence: {feature}')
    plt.tight_layout()
    plt.show()

## Clinical Interpretation Notes

- SHAP values explain the model, not causal biology. A high contribution means the model used that feature strongly for this prediction.
- Global rankings identify the strongest model drivers across the test set.
- Waterfall plots are useful for patient-level conversations because they show which inputs raised or lowered a single risk score.
- These explanations are educational and should not be used as medical advice without clinician review.